# AirSketch — Results Visualization
Loads all finished W&B runs from `airsketch/AirSketch` and plots training curves, sweep heatmaps, and best-run summaries.

**Run this notebook after at least one training run has completed.**

In [ ]:
# Cell 1 — Authenticate
import wandb
import os

# On Zaratan, WANDB_API_KEY is set in the environment.
# Locally, either set the env var or call wandb.login() interactively.
if not os.environ.get('WANDB_API_KEY'):
    wandb.login()
else:
    print('Using WANDB_API_KEY from environment.')

In [ ]:
# Cell 2 — Load all finished runs
import pandas as pd

api = wandb.Api()
runs = api.runs('airsketch/AirSketch')

records = []
for run in runs:
    if run.state != 'finished':
        continue
    records.append({
        'name':        run.name,
        'hidden_dim':  run.config.get('model', {}).get('hidden_dim'),
        'num_layers':  run.config.get('model', {}).get('num_layers'),
        'lr':          run.config.get('training', {}).get('learning_rate'),
        'val_mpjpe':   run.summary.get('val/mpjpe'),
        'val_jitter':  run.summary.get('val/jitter_index'),
        'val_gesture': run.summary.get('val/gesture_acc'),
        'best_epoch':  run.summary.get('epoch'),
    })

df = pd.DataFrame(records).sort_values('val_mpjpe')
print(f'Loaded {len(df)} finished runs.')
df

In [ ]:
# Cell 3 — Training curves for all runs
import matplotlib.pyplot as plt
import os

os.makedirs('report/figures', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ['val/mpjpe', 'val/jitter_index', 'val/gesture_acc']
titles  = ['Val MPJPE (px) ↓', 'Val Jitter Index (px²) ↓', 'Val Gesture Acc ↑']
targets = [8.0, 2.0, 0.92]

for ax, metric, title, target in zip(axes, metrics, titles, targets):
    for run in runs:
        if run.state != 'finished':
            continue
        history = run.history(keys=[metric])
        if metric in history.columns:
            ax.plot(history[metric].values, alpha=0.6, label=run.name)
    ax.axhline(y=target, color='red', linestyle='--', linewidth=1.5, label=f'Target: {target}')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7, loc='best')

plt.tight_layout()
plt.savefig('report/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Sweep heatmap (run after issue #11 sweep completes)
import seaborn as sns

pivot = df.pivot_table(
    index='num_layers',
    columns='hidden_dim',
    values='val_mpjpe',
    aggfunc='min',
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    pivot,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd_r',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Val MPJPE (px)'},
)
ax.set_title('Sweep results — val MPJPE by hidden_dim × num_layers\n(lower is better, target < 8 px)')
ax.set_xlabel('Hidden dim')
ax.set_ylabel('Num layers')
plt.tight_layout()
plt.savefig('report/figures/sweep_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Best run summary
best = df.iloc[0]
print('Best run:')
print(f"  Name:        {best['name']}")
print(f"  hidden_dim:  {best['hidden_dim']}")
print(f"  num_layers:  {best['num_layers']}")
print(f"  lr:          {best['lr']}")
print(f"  val MPJPE:   {best['val_mpjpe']:.3f} px  (target < 8 px)")
print(f"  val jitter:  {best['val_jitter']:.3f} px² (target < 2 px²)")
print(f"  val gesture: {best['val_gesture']:.3f}    (target > 0.92)")